In [1]:
import os
import sys
import pandas as pd
import json
import binascii
from pathlib import Path
import subprocess
import base64
import sqlalchemy
ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
ETL_Path = ROOT / "ETL"
sys.path.insert(0, str(ETL_Path))
from src.read_write_utils import (
    split_to_parquet,
    combine_and_read_parquet,
    load_to_postgres
)


# Destination folder
MQTT_PARTS_GZ_FOLDER = ROOT / "Data"/ "Landing_Zone" / "emf-mqtt-dump"
print(MQTT_PARTS_GZ_FOLDER.resolve())

C:\Users\Graem\Desktop\Github Repos\EMF_Camp_DataEng\Data\Landing_Zone\emf-mqtt-dump


## combine and read files. 
Need to check this soltution has no effects on data integrety. 

In [2]:
gzip_files = sorted(
    MQTT_PARTS_GZ_FOLDER.glob("mqtt.emf.camp.csv.gz.*"),
    key=lambda path: int(path.name.rsplit(".", 1)[-1])
)

if len(gzip_files) != 2:
    raise ValueError(f"Expected 2 gzip parts, found {len(gzip_files)}: {gzip_files}")

combined_gzip = MQTT_PARTS_GZ_FOLDER / "combined_csv.gz"

with combined_gzip.open("wb") as output:
    for file in gzip_files:
        with file.open("rb") as part:
            while chunk := part.read(1024 * 1024):
                output.write(chunk)


In [3]:
columns = ["msg.topic", "payload", "msg.qos", "msg.retain", "time_now", "time_delta"]
df = pd.read_csv(combined_gzip, compression="gzip", header=None, names=columns)

print(f"Combined GZIP saved to: {combined_gzip}")

df.head()


Combined GZIP saved to: C:\Users\Graem\Desktop\Github Repos\EMF_Camp_DataEng\Data\Landing_Zone\emf-mqtt-dump\combined_csv.gz


,msg.topic,payload,msg.qos,msg.retain,time_now,time_delta
0,open/jungle-breaks-served,NDE=,0,False,1.784160e+09,1.699473
1,open/wnstlyktrasdi,MDExMDExMTE=,0,False,1.784160e+09,1.627660
2,open/wnstlyktrasdi,MDExMTAxMDE=,0,False,1.784160e+09,4.000714
3,open/view-the-full-schedule-timeline-at,aHR0cHM6Ly9oZWFsdy5pbmc=,0,False,1.784160e+09,0.781992
4,open/hackers,eyJTY2VuZS1sb2NhdGlvbiI6ICJJTlQuIFBMQUdVRSdTIE...,0,False,1.784160e+09,0.005430


## Decode Payload and time

In [4]:
# Decode Base64 

def decode_payload(value):
    if pd.isna(value):
        return None

    try:
        # Decode Base64
        return base64.b64decode(value, validate=True).decode("utf-8")

    except (ValueError, UnicodeDecodeError, binascii.Error):
        # binascii.Error:  When the input contains invalid Base64 data. For example, incorrectly formatted Base64 strings.
        # UnicodeDecodeError: decoded bytes cannot be interpreted as valid UTF-8 text.
        # ValueError: Handles other invalid input values.
        return value

df["payload"] = df["payload"].map(decode_payload)

# Convert Unix time (Standard derived from an absolute point in time, measured from 1 January 1970 UTC.)
df["time_now"] = pd.to_datetime(
    df["time_now"], unit="s", utc=True, errors="coerce"
).dt.tz_convert("Europe/London")


display(df)

,msg.topic,payload,msg.qos,msg.retain,time_now,time_delta
0,open/jungle-breaks-served,41,0,False,2026-07-16 01:00:01.638832092+01:00,1.699473
1,open/wnstlyktrasdi,01101111,0,False,2026-07-16 01:00:03.266491890+01:00,1.627660
2,open/wnstlyktrasdi,01110101,0,False,2026-07-16 01:00:07.267206192+01:00,4.000714
3,open/view-the-full-schedule-timeline-at,https://healw.ing,0,False,2026-07-16 01:00:08.049198389+01:00,0.781992
4,open/hackers,"{""Scene-location"": ""INT. PLAGUE'S LOFT."", ""Sce...",0,False,2026-07-16 01:00:08.054627895+01:00,0.005430
...,...,...,...,...,...,...
1927245,power/generator/s,"{""gencomm_version"": 2.0, ""password_status"": 3....",0,False,2026-07-20 12:59:57.016425133+01:00,1.005037
1927246,power/generator/s,"{""gencomm_version"": 2.0, ""password_status"": 3....",0,False,2026-07-20 12:59:58.011878967+01:00,0.995454
1927247,weather/hq,"{""runtime"": 878378.0, ""heap"": 139448.0, ""tempi...",0,False,2026-07-20 12:59:58.341118813+01:00,0.329240
1927248,power/generator/s,"{""gencomm_version"": 2.0, ""password_status"": 3....",0,False,2026-07-20 12:59:59.011588335+01:00,0.670470


# Save Checkpoint 
Parquet broken up to ensure ability to load to github

In [5]:
BR_MQTT_DATA = ROOT / "Data" / "Bronze" / "br_MQTT"

split_to_parquet(
    df = df,
    directory= BR_MQTT_DATA,
    target_size_mb=50
)

testdf = combine_and_read_parquet(directory=BR_MQTT_DATA)
assert testdf.equals(df), "Something went wrong in the read/write process"

# Load Data to postgres

In [34]:
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

DB_NAME = os.getenv('DB_NAME')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_USER = os.getenv('DB_USER')

connection_uri = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

load_to_postgres(
    df=df,
    connection_string= connection_uri,
    table_name= "br_mqtt_all",
    schema="public",
    chunksize=5000
    )

### Confirm SQL data. 
Should revise this code block

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

DB_NAME = os.getenv('DB_NAME')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_USER = os.getenv('DB_USER')

connection_uri = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
db_engine = sqlalchemy.create_engine(connection_uri)

sql_test = pd.read_sql(
    "SELECT * FROM public.br_mqtt_all",
    con=db_engine
)

assert len(sql_test) == len(df), "Data was lost in upload"

################################################################################
######## Should replace this section beneath with well understood testing rather than vibed troubelshooting code to test. 
################################################################################
if not df.equals(sql_test):
    df_check = df.copy()
    sql_check = sql_test.copy()

    if not df.equals(sql_test):
        df_check = df.copy()
        sql_check = sql_test.copy()

        # Compare timestamps in UTC at PostgreSQL's microsecond precision
        for frame in (df_check, sql_check):
            frame["time_now"] = pd.to_datetime(
                frame["time_now"], utc=True
            ).dt.floor("us")

        # SQL queries do not guarantee row order
        sort_columns = list(df_check.columns)
        df_check = df_check.sort_values(sort_columns, na_position="last").reset_index(drop=True)
        sql_check = sql_check.sort_values(sort_columns, na_position="last").reset_index(drop=True)

        try:
            pd.testing.assert_frame_equal(
                df_check,
                sql_check,
                check_dtype=False,
                check_exact=False,
                rtol=1e-9,
                atol=1e-12,
            )
            print("Data matches after accounting for row order and timestamp precision.")
        except AssertionError as exc:
            print("Data differs:")
            print(exc)

            unequal = df_check.ne(sql_check) & ~(df_check.isna() & sql_check.isna())
            bad_rows = unequal.any(axis=1)
            print(f"Found {bad_rows.sum()} mismatched rows. Showing up to 20:")
            display(pd.concat(
                {"df": df_check, "sql": sql_check}, axis=1
            ).loc[bad_rows].head(20))

    # SQL queries do not guarantee row order
    sort_columns = list(df_check.columns)
    df_check = df_check.sort_values(sort_columns, na_position="last").reset_index(drop=True)
    sql_check = sql_check.sort_values(sort_columns, na_position="last").reset_index(drop=True)

    try:
        pd.testing.assert_frame_equal(
            df_check,
            sql_check,
            check_dtype=False,
            check_exact=False,
            rtol=1e-9,
            atol=1e-12,
        )
        print("Data matches after accounting for row order and timestamp precision.")
    except AssertionError as exc:
        print("Data differs:")
        print(exc)


Data matches after accounting for row order and timestamp precision.


## Examine topics 

In [ ]:
db_engine = sqlalchemy.create_engine(connection_uri)
top_level_topics = pd.read_sql("""
    SELECT
    SPLIT_PART("msg.topic", '/', 1) AS Topic,
	SPLIT_PART("msg.topic", '/', 2) AS Subtopic,
    COUNT(*) AS message_count
    FROM public.br_mqtt_all
    GROUP BY Topic , Subtopic
    ORDER BY message_count DESC
    LIMIT 50;
    """
    ,con=db_engine
)
print("\n\narranged by message count")
display(top_level_topics)

print("\n\n\n Arranged by topics")

org_by_cat = top_level_topics.sort_values(
    by=["topic", "subtopic"],
    ascending=[True, True]
)
display(org_by_cat)



arranged by message count


,topic,subtopic,message_count
0,emf,arcade,890655
1,power,generator,148865
2,open,wnstlyktrasdi,80666
3,open,map,70061
4,lorawan,application,61034
5,phones,total-calls,46316
6,phones,calls-answered,36441
7,phones,answer-rate,36441
8,phones,numbers-assigned,36440
9,phones,voicemail-messages,36440





 Arranged by topics


,topic,subtopic,message_count
31,airgradient,readings,3785
43,arcade,cab-14,7
41,arcade,cab-15,14
47,arcade,cab-16,7
42,arcade,cab-4,14
46,arcade,cab-7,7
45,arcade,cab-9,7
20,bar,stockitem,21989
24,bar,stockline,14242
21,bar,stocktype,21651
